**Data Export & PostgreSQL Ingestion**

Cleaned and featured datasets were exported from pandas (csv files) and loaded into PostgreSQL (tabels) for analytical querying and BI usage.

**Revenue & Time Trends**

**Business Questions**

How business performs over time?

Specifically:
- How does revenue evolve month over month?
- Is growth driven by more orders or higher order value?
- Are there visible trends or volatility patterns?

Data Used

Table:
- public.cleaned_transactions

Reminder:
- One row = one invoice line
- Order-level metrics must use COUNT(DISTINCT order_id)

Revenue definition:
- line_revenue already includes returns (negative values)

All metrics below are net revenue

In [ ]:
########## Monthly Revenue Trends

# Purpose 
# Understand Overall Business Performace and Seasonality

# SELECT 
# 	DATE_TRUNC('month', order_date) AS month,
# 	SUM(line_revenue) AS total_revenue
# FROM
# 	cleaned_transactions
# GROUP BY 1
# ORDER BY 1

# Interpretation:
# - Shows net revenue per calendar month
# - Returns automatically reduce revenue

# Used later for:
# - trend charts
# - executive KPIs
# - forecasting discussions

########## Monthly Order Volume

# Purpose
# Separate volume growth from value growth

# SELECT 
# 	DATE_TRUNC('month', order_date) AS month,
# 	COUNT(DISTINCT order_id) AS total_orders
# FROM
# 	cleaned_transactions
# GROUP BY 1
# ORDER BY 1

# Interpretation:
# - Counts completed orders only (cancellations already removed)

# Helps answer:
# - Are we selling more orders?
# - Or just more expensive ones?

########## Monthly Average Order Value

# Purpose 
# Measure customer spending behavior over time

# SELECT 
# 	DATE_TRUNC('month', order_date) AS month,
# 	SUM(line_revenue) / COUNT(DISTINCT order_id) AS AOV
# FROM
# 	cleaned_transactions
# GROUP BY 1
# ORDER BY 1

# Interpretation:
# Net AOV (returns included)

# Sensitive to:
# - pricing changes
# - discounts
# - product mix shifts


########## Month-Over-Month Revenue Change

# Purpose
# Detect growth, decline, volatility
# WITH monthly_revenue AS (
#  SELECT
#         DATE_TRUNC('month', order_date) AS month,
#         SUM(line_revenue)               AS revenue
#     FROM cleaned_transactions
#     GROUP BY 1
# )
# SELECT
#     month,
#     revenue,
#     revenue - LAG(revenue) OVER(ORDER BY MONTH) AS mom_revenue_change,
#     (revenue - LAG(revenue) OVER(ORDER BY MONTH)) / NULLIF(LAG(revenue) OVER(ORDER BY MONTH),0)
#     AS mom_growth_rate
# FROM monthly_revenue
# ORDER BY MONTH

# Interpretation:
# - Absolute change highlights financial impact
# - Growth rate highlights relative performance
# - NULLIF avoids division-by-zero errors

########## Revenue vs Order Growth (Combined View)

# Purpose 
# Understand why revenue changes

# SELECT 
# 	DATE_TRUNC('month', order_date) AS month,
# 	SUM(line_revenue) AS total_revenue,
#   COUNT(DISTINCT order_id) AS total_orders,
#   SUM(line_revenue) / COUNT(DISTINCT order_id) AS AOV
# FROM
# 	cleaned_transactions
# GROUP BY 1
# ORDER BY 1

# Interpretation:

# One table answers:
# - revenue trend
# - volume trend
# - value trend

# This table is perfect for Power BI fact table import

**These metrics will help to asnwer on such questions:**

Is revenue trend stable, growing, or volatile?
Are revenue spikes driven by:
- more orders?
- higher AOV?
- Are there periods of decline that need explanation?



**Customer Ranking and Value**

Business Question

Who are best and worst customers, and how is customer value distributed?

Specifically:
- Which customers generate the most revenue?
- Who purchases most frequently?
- Who has the highest average order value?
- How concentrated is customer value?

**Data Used**

Primary table:
- public.customer_summary

Why this table:
- One row per customer
- Metrics already aggregated correctly
- No risk of double counting

In [ ]:
########## Rank Customers by Total Revenue

# Purpose
# Identify the most valuable customers

# SELECT customer_id,
# total_revenue,
# RANK(total_revenue) OVER(ORDER BY total_revenue DESC) AS revenue_rank
# FROM customer_summary
# ORDER BY revenue_rank
# LIMIT 20

# Interpretation:

# - Shows top revenue-generating customers
# - Uses RANK() to handle ties transparently
# - Suitable for VIP identification


########## Rank Customers by Number of Orders

# Purpose
# Identify high-frequency buyers

# SELECT customer_id,
# total_orders,
# RANK(total_orders) OVER(ORDER BY total_orders DESC) AS order_frequency_rank
# FROM customer_summary
# ORDER BY order_frequency_rank
# LIMIT 20

# Interpretation:

# - Highlights loyalty and engagement
# - Often overlaps with top revenue customers, but not always


########## Rank Customers By AOV

# Purpose
# Idetify who placed high-value orders, even i infrequently

# SELECT customer_id,
# avg_order_value,
# RANK(avg_order_value) OVER(ORDER BY avg_order_value DESC) AS aov_rank
# FROM customer_summary
# ORDER BY aov_rank
# LIMIT 20

# Interpretation:

# - Useful for premium segmentation
# - Complements frequency-based rankings


########## Combined Customer Ranking View

# Purpose
# Compare different dimension of customer value

# SELECT customer_id,
# total_revenue,
# total_orders,
# avg_order_value,
# RANK(total_revenue) OVER(ORDER BY total_revenue DESC) AS revenue_rank,
# RANK(total_orders) OVER(ORDER BY total_orders DESC) AS order_frequency_rank,
# RANK(avg_order_value) OVER(ORDER BY avg_order_value DESC) AS aov_rank
# FROM customer_summary
# ORDER BY revenue_rank
# LIMIT 50

# Interpretation:

# Shows whether top customers:
# - buy often
# - buy big
# - or both

# Strong foundation for customer segmentation


########## Revenue Contribution per Customer (% of Total)

# Purpose
# Quantify relative customer importance

# SELECT
#     customerid,
#     total_revenue,
#     total_revenue
#         / SUM(total_revenue) OVER () AS revenue_share
# FROM customer_summary
# ORDER BY total_revenue DESC
# LIMIT 20

# Interpretation:
# - Shows how much each customer contributes to total revenue
# - Direct input for Pareto analysis (next section)

########## Identify Low Value customers

# SELECT
#     COUNT(*) AS low_value_customers
# WHERE total_revenue < 100
# FROM customer_summary

# Interpretation:
# Quantifies customers with minimal impact

# Important for:
# - cost-to-serve discussions
# - segmentation decisions

**Repeat Purchase and Frequency Analysis**

Business question

How loyal are our customers?

Specifically:

- How many customers purchase only once vs repeatedly?
- What is the typical number of orders per customer?
- How frequently do customers return to make another purchase?


Data Used

Primary tables:

- public.customer_summary
- public.cleaned_transactions (for time gaps)

In [ ]:
########## Distribution of orders per customer

# Purpose
# Understand how many customers are one-time buyers vs repeat buyers

# SELECT total_orders,
# COUNT(*) AS customers
# FROM customer_summary
# GROUP BY total_orders
# ORDER BY total_orders

# Interpretation:
# - Shows the full distribution of customer frequency
# - Typically, e-commerce has a large one-time buyer group



########## One-Time vs Repeat Customers

# Purpose
# Quantify customer loyalty at high level

# SELECT 
#     CASE
#         WHEN total_orders = 1 THEN 'One-time'
#         ELSE 'Repeat'
#     END AS customer_type,
#     COUNT(*) AS customers
# FROM customer_summary
# GROUP BY 1

# Interpretation:
# - Clear split between transactional and loyal customers
# - Useful for executive-level KPIs


########## Share of Revenue per Customer Type

# Purpose
# compare customer count vs customer distribution


# SELECT 
#     CASE
#         WHEN total_orders = 1 THEN 'One-time'
#         ELSE 'Repeat'
#     END AS customer_type,
#     SUM(total_revenue) AS customers
# FROM customer_summary
# GROUP BY 1

# Interpretation:

# Often repeat customers generate disproportionate revenue
# Highlights importance of retention



########## Average Orders per Customer

# Purpose
# Summarize purchase frequency with a single metric

# SELECT AVG(total_orers)::NUMERIC(10,2) AS avg_orders_per_customer
# FROM customer_summary

# Interpretation:
# - High-level indicator of customer engagement
# - Useful for benchmarking over time or across markets


########## Time Between Orders (Customer-Level)

# Purpose
# Understand how frequently customers return.

# WITH customer_orders AS (
#     SELECT
#         customerid,
#         order_id,  
#         MIN(order_date),
#     FROM cleaned_transactions
#     WHERE has_customer = true
#     GROUP BY customerid, order_id  
# ),
# order_gaps AS (
#     SELECT
#         customerid,
#         order_date,
#         LAG(order_date) OVER (
#             PARTITION BY customerid
#             ORDER BY order_date
#         ) AS previous_order_date
#     FROM customer_orders
# )
# SELECT
#     AVG(order_date - previous_order_date) AS avg_days_between_orders
# FROM order_gaps
# WHERE previous_order_date IS NOT NULL;


# Interpretation:
# - Measures average repurchase interval
# - Lower values indicate stronger engagement


########## Distribution of Days between orders

# Purpose
# Detect variability in customer behavior


# WITH customer_orders AS (
#     SELECT
#         customerid,
#         order_id,  
#         MIN(order_date),
#     FROM cleaned_transactions
#     WHERE has_customer = true
#     GROUP BY customerid, order_id  
# ),
# order_gaps AS (
#     SELECT
#         customerid,
#         order_date,
#         LAG(order_date) OVER (
#             PARTITION BY customerid
#             ORDER BY order_date
#         ) AS previous_order_date
#     FROM customer_orders
# )
# SELECT
#     (order_date - previous_order_date) AS days_between_orders,
#     COUNT(*) AS occurrences
# FROM order_gaps
# WHERE previous_order_date IS NOT NULL
# GROUP BY 1
# ORDER BY 1

# 


**Pareto (80/20) Analysis**

Business Question

How concentrated is our revenue?

Specifically:

- What percentage of customers generates the majority of revenue?
- Is the business dependent on a small group of high-value customers?
- How large is the long tail of low-impact customers?

**Data Used**

Table:
- public.customer_summary

In [ ]:
########## Customer Revenue Ranking

# Purpose
# Order Customers from Highest to Lowest

# SELECT
#     customerid,
#     total_revenue
# FROM customer_summary
# ORDER BY total_revenue DESC;

# This ordered list is the foundation for cumulative calculations.

########## Cumulative Revenue by Customer

# Purpose
# Calculate how revenue accumulates as we move down the customer list

# SELECT
#     customerid,
#     total_revenue,
#     SUM(total_revenue) OVER(ORDER BY total_revenue DESC
#         ROWS BETWEEN UNBONUNDED PRECEDING AND CURRENT ROW
#     ) AS culumative_total_revenue
# FROM customer_summary
# ORDER BY total_revenue DESC;

# Interpretation:
# - Each row shows revenue generated by the top N customers
# - Used to measure concentration


########## Cumulative Revenue Share (%)

# Purpose
# Express cumulative revenue as the percentage of total revenue

# SELECT
#     customerid,
#     total_revenue,
#     SUM(total_revenue) OVER(ORDER BY total_revenue DESC
#         ROWS BETWEEN UNBONUNDED PRECEDING AND CURRENT ROW
#     ) / SUM(total_revenue) OVER() AS culumative_revenue_share
# FROM customer_summary
# ORDER BY total_revenue DESC;

# Interpretation:
# When cumulative_revenue_share reaches 0.80, we have identified the Pareto threshold
# This directly answers: “How many customers generate 80% of revenue?”

########## Identify Customers Driving 80% of Revenue

# Purpose
# Check the Pareto's group size

# WITH pareto AS(
    # SELECT
#     customerid,
#     total_revenue,
#     SUM(total_revenue) OVER(ORDER BY total_revenue DESC
#         ROWS BETWEEN UNBONUNDED PRECEDING AND CURRENT ROW
#     ) / SUM(total_revenue) OVER() AS culumative_revenue_share
# FROM customer_summary
# )
# SELECT COUNT(*) FROM pareto
# WHERE culumative_revenue_share <= 0.8

# Interpretation:
# - Small number → high dependency risk
# - Larger number → more diversified revenue base

########## Share of Customers vs Share of Revenue

# Purpose 
# Produce Poreto curve for PowerBI

# WITH ranked_customers AS (
#     SELECT
#         customerid,
#         total_revenue,
#         ROW_NUMBER() OVER (ORDER BY total_revenue DESC) AS customer_rank,
#         COUNT(*) OVER () AS total_customers,
#         SUM(total_revenue) OVER () AS total_revenue_all
#     FROM customer_summary
# )
# SELECT
#     customer_rank::NUMERIC / total_customers AS customer_percentile,
#     SUM(total_revenue) OVER (
#         ORDER BY customer_rank
#         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
#     ) / total_revenue_all AS cumulative_revenue_share
# FROM ranked_customers
# ORDER BY customer_percentile;

# Interpretation:

# - X-axis: % of customers
# - Y-axis: % of revenue
# - Perfect input for a Pareto line chart

########## Top vs Long-Tail Revenue Split

# Compare Top Customers vs Rest

# WITH pareto AS (
#     SELECT
#         customerid,
#         total_revenue,
#         SUM(total_revenue) OVER (
#             ORDER BY total_revenue DESC
#         )
#             / SUM(total_revenue) OVER () AS cumulative_revenue_share
#     FROM customer_summary
# )
# SELECT
#     CASE
#         WHEN cumulative_revenue_share <= 0.80 THEN 'Top 80% revenue group'
#         ELSE 'Remaining customers'
#     END AS customer_segment,
#     COUNT(*) AS customers,
#     SUM(total_revenue) AS revenue
# FROM pareto
# GROUP BY 1;

# Interpretation:

# - Shows imbalance between customer count and revenue share
# - Very strong executive-level insight

**Returns Impact Analysis**

Business question
Where are we losing money due to returns

Specifically:
- How much revenue is lost due to returns?
- What is iverall return rate?
- Which products and customers drive returns?
- How large is the gap between gross and net revenue?

Data Used

Primary table:
public.cleaned_transactions

Key fields:
- line_revenue (negative for returns)
- is_return_revenue (true for returned lines)
- order_id, stockcode, customerid

In [ ]:
########## Gross vs Net Revenue

# Purpose 
# Quintify revenue erosion cause by returns

# SELECT
#     SUM(CASE line_revenue > 0 THEN line_revenue ELSE 0) AS gross_revenue,
#     SUM(CASE line_revenue < 0 THEN line_revenue ELSE 0) AS return_revenue,
#     SUM(line_revenue) AS net_revenue,
# FROM clean_transactions

# Interpretation:
# - gross_revenue = sales before returns
# - return_revenue = revenue lost to returns (negative)
# -net_revenue = true realized revenue

########## Overall return rate

# Purpose
# Measure the scale of returns relative to sales

# SELECT ABS(SUM(CASE line_revenue < 0 THEN line_revenue ELSE 0) 
# / 
# NULLIF(SUM(CASE line_revenue > 0 THEN line_revenue ELSE 0),0)) AS overal_return_rate
# FROM cleaned_transactions

# Interpretation:
# - Percentage of sales revenue that is returned
# - High values indicate operational or product issues

########## Return Rate By Product

# Purpose
# Identify products with abnormal return behavior

# SELECT
#     stockcode,
#     SUM(CASE WHEN line_revenue > 0 THEN line_revenue ELSE 0 END) AS gross_revenue,
#     ABS(SUM(CASE WHEN line_revenue < 0 THEN line_revenue ELSE 0 END)) AS return_revenue,
#     ABS(SUM(CASE WHEN line_revenue < 0 THEN line_revenue ELSE 0 END))
#         / NULLIF(SUM(CASE WHEN line_revenue > 0 THEN line_revenue ELSE 0 END), 0)
#         AS return_rate
# FROM cleaned_transactions
# GROUP BY stockcode
# HAVING SUM(CASE WHEN line_revenue > 0 THEN line_revenue ELSE 0 END) > 0
# ORDER BY return_rate DESC
# LIMIT 20;

# Interpretation
# - Highlights products with disproportionately high return rates
# - Useful for quality control and supplier review

########## Customers With High Return Impact

# Purpose
# Indentify customers whose purchases generate high return losses

# SELECT
#     customerid,
#     SUM(CASE WHEN line_revenue > 0 THEN line_revenue ELSE 0 END) AS gross_revenue,
#     ABS(SUM(CASE WHEN line_revenue < 0 THEN line_revenue ELSE 0 END)) AS return_revenue,
#     ABS(SUM(CASE WHEN line_revenue < 0 THEN line_revenue ELSE 0 END))
#         / NULLIF(SUM(CASE WHEN line_revenue > 0 THEN line_revenue ELSE 0 END), 0)
#         AS return_rate
# FROM cleaned_transactions
# WHERE has_customer = true
# GROUP BY customerid
# HAVING SUM(CASE WHEN line_revenue > 0 THEN line_revenue ELSE 0 END) > 0
# ORDER BY return_rate DESC
# LIMIT 20;

# Interpretation:
# - Flags customers with unusually high return behavior
# - Useful for fraud detection, policy review, or customer segmentation


########## Orders Dominated by Returns

# Purpose
# Detect problematic orders, where returns outweigh sales

# WITH sub_query AS (
# SELECT
#     order_id,
#     SUM(line_revenue) AS order_net_revenue
# FROM 
#     cleaned_transactions
# GROUP BY order_id
# )
# SELECT COUNT(*) AS net_negative_orders
# FROM sub_query 
# WHERE order_net_revenue < 0

# Interpretation:
# - Orders that cost more than they generated
# - Indicates fulfillment, logistics, or customer behavior issues

########## Return Impact Over Time

# Purpose
# Indicate whether returns are increasing or stable

# SELECT
#     DATE_TRUNC('month', order_date) AS month,
#     SUM(CASE WHEN line_revenue > 0 THEN line_revenue ELSE 0 END) AS gross_revenue,
#     ABS(SUM(CASE WHEN line_revenue < 0 THEN line_revenue ELSE 0 END)) AS return_revenue,
#     ABS(SUM(CASE WHEN line_revenue < 0 THEN line_revenue ELSE 0 END))
#         / NULLIF(SUM(CASE WHEN line_revenue > 0 THEN line_revenue ELSE 0 END), 0)
#         AS return_rate
# FROM cleaned_transactions
# GROUP BY 1
# ORDER BY 1

# Interpretation:
# - Reveals trends in return behavior
# - Critical for operational monitoring

**Time-Based Customer Behavior**

Business Question

How does customer activity evolve over time

Specifically
How long do customers stay active?
How many customers are inactive?
What does the customer lifecycle look like?
How large is the risk of customer churn?

Data Used

Primary table:
- public.customer_summary

Supporting table (order timing):
- public.cleaned_transactions

In [ ]:
########## Customer Lifecycle Length

# Purpose
# Measure how long customers remain active

# SELECT 
#     customer_lifetime_days,
#     COUNT(*) AS customers
# FROM 
#     customer_summary
# GROUP BY customer_lifetime_days

# Interpretation:
# - Short lifetime - transactional customers
# - Long lifetime - loyal customers
# - Skewed distributions are normal in e-commerce



########## Average & Median Customer Lifetime

# Purpose
# Summarize lifecycle duration

# SELECT
#     AVG(customer_lifetime_days),
#     PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY customer_lifetime_days) AS median_lietime_days
# FROM 
#     customer_summary

# Interpretation
# - median is more informative than mean
# - highlights whether long-tail customers distort averages


########## Last Purchase Recency per Customer

# Purpose
# Measure how recently customers were active

# SELECT
#     CURRENT_DATE - last_purchase AS days_since_last_purchase
# FROM 
#     customer_summary

# Interpretation
# Core metrics for retention and churn analysis
# Used later for inactive customer classification


# Identify Inactive Customers (Time-Based)

# Purpose
# Detect Customers who have likely churned
# Define inactivity theshold
# No purchase in last 90 days

# SELECT
#     COUNT(*) AS inactive_customers
# FROM 
#     customer_summary
# WHERE
#     last_purchase < CURRENT_DATE - Interval '90 days'

# Interpretation
# Gives a high-level churn risk indicator
# Treshold can be adjusted per business context

########## Active & Inactive Customer Split

# Purpose
# Compare count of active and inactive users

# SELECT
#     CASE
#         WHEN last_purchase < CURRENT_DATE - INTERVAL '90 days'
#             THEN 'Inactive'
#         ELSE 'Active'
#     END AS customer_status,
#     COUNT(*) AS customers
# FROM customer_summary
# GROUP BY 1;

# Interpretation
# - reveals retention halth
# - useful for dashboards

########## Revenue Contribution by Activity Status

# Purpose
# Understand Revenue Risk From inactivity

# SELECT
#     CASE
#         WHEN last_purchase < CURRENT_DATE - INTERVAL '90 days'
#             THEN 'Inactive'
#         ELSE 'Active'
#     END AS customer_status,
#     SUM(total_revenue) AS revenue
# FROM customer_summary
# GROUP BY 1;

# Interpretation
# - If inactive customers contributed large revenue historically - high risk
# - Strong justification for reactivation campaigns

########## Customer Activity over Time

# Purpose
# Visualize customer lifecycle spread

# SELECT 
#     DATE_TRUNC('month', first_purchase) AS cohort_month,
#     COUNT(*) AS customers_amount
# FROM customer_summary
# GROUP BY 1
# ORDER BY 1;

# Interpretation
# - This is pre-cohort table
# Used later for 
# - cohort analysis
# - retention metrics in PowerBI

**Analytical SQL COMPLETE**

At this point we have:

- revenue trends
- customer rankings
- repeat behavior analysis
- Pareto concentration
- returns impact
- time-based customer behavior